# PKG edge-state benchmark — what will each relationship pay next month? (v1.1)

**Why.** The contagion study defines a shock with fixed rules (a steady payer falling 50% short). Looser or stricter
rules count different events, and the rule ignores how often a payer pays. This notebook asks the prior question:
which way of reading a payment relationship best predicts what it pays next month? The best one can then define the
shock as a *surprise* (came − expected) for every relationship, steady or not, and is the first piece of a model of
the whole network one month ahead.

**Models compared** (§4): last month, 3-month average, the study's steady rule, the periodicity tables, BG/BB
(discrete-time buy-till-you-die: a relationship active 11 of 12 months and silent now is more likely over than one
active 4 of 6), a logistic model on the history, the same plus BG/BB, and the same plus network inputs (is the payer
missing other PNC payees, is the payer itself shocked, is the customer's income falling).

**v1.1 (2026-11-16):** streams from other PNC customers were dropped in v1.0 (their own-account flag is empty and
the filter treated empty as "own account"): fixed, so PNC payers and the upstream input are in. Adds an amount model
(next month's amount when it pays), a 3-month surprise scaled by its uncertainty (§5), and the same surprise for the
whole population written for the contagion study's definition grid (§6).

**Sections.** §1 inputs and sample · §2 stream-month panel with features and next-month targets · §3 BG/BB fit per
segment · §4 scores on later months (AUC, log loss, calibration, dollar error per stream and per customer, by history
class) · §5 surprise over 3 months as a candidate event · §6 surprise for the whole population · §7 answer sheet.

Run all. One output per cell; csv files in `OUT_DIR`; work files in `WORK_DIR` (reused on a rerun with the same
settings, set `REBUILD = True` to rebuild).

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import os, sys, time, json, math, hashlib, warnings, threading, contextlib
import numpy as np
import pandas as pd

for _a, _t in [("object0", object), ("bool8", np.bool_), ("int0", np.intp), ("uint0", np.uintp), ("bool", np.bool_)]:
    if not hasattr(np, _a):
        setattr(np, _a, _t)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _w in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_w)

from pyspark.sql import SparkSession, functions as F, Window as W

DB = "bdahd01p_dlcdi1_cdi_tm"
CONFIG = dict(
    PAIR_TABLE     = f"{DB}.pkg_cust_pair_month_v4",   # cust, dir, scope, other, amount, o_self, month
    DIM_TABLE      = f"{DB}.pkg_cust_dim",             # mdm_id, is_tm, party_type, pwr_shared
    REG_STREAM     = f"{DB}.pkg_cust_reg_stream",      # periodicity v1.1 (optional; None to skip)
    ORG_PARTY_TYPES = ["O"],
    LAST_MONTH     = None,            # None = latest month of the pair table
    FEATURE_START_AFTER = 12,         # months of history before the first as-of month
    TRAIN_END      = "2025-10",       # as-of months up to here train the models; later ones test them
    SAMPLE_PCT     = 10,              # % of TM organisations whose streams are modelled (hash of mdm id)
    AMOUNT_BANDS   = [10_000, 100_000, 1_000_000],   # usual paid amount bands for BG/BB segments
    BGBB_MIN_STREAMS = 2_000,         # smaller segments use the parameters fitted on all streams
    TRAIN_ROWS     = 1_500_000,       # training stream-months brought to pandas
    TEST_ROWS      = 1_500_000,       # test stream-months (whole customers, so customer totals are complete)
    REBUILD        = False,
    BUILD_SURPRISE_ALL = True,        # §6: stream surprise for every TM organisation (for the contagion study §5h)
    SURPRISE_FROM  = "2024-11",       # first as-of month (events 2 months later: 2025-01)
    SURPRISE_OUT   = "/user/pk36814/pkg_work/edge_state/surprise_streams",   # parquet read by pkg_contagion_p1 §5h
    SHUFFLE_PARTITIONS = 800,
    WORK_DIR       = "/user/pk36814/pkg_work/edge_state",   # HDFS folder for work files
    OUT_DIR        = "../metrics/edge_state_benchmark",
)

spark = (SparkSession.builder.appName("pkg_edge_state_benchmark")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "8g").config("spark.executor.memoryOverhead", "3g")
         .config("spark.executor.cores", "2").config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g").config("spark.dynamicAllocation.maxExecutors", "256")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.shuffle.partitions", str(CONFIG["SHUFFLE_PARTITIONS"]))
         .enableHiveSupport().getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:
    _L = spark._jvm.org.apache.logging.log4j
    for _n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage", "org.apache.kafka",
               "org.apache.spark.sql.catalyst.util.package", "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    pass
spark.conf.set("spark.sql.shuffle.partitions", str(CONFIG["SHUFFLE_PARTITIONS"]))

try:
    from IPython.display import display, HTML
    _NB = "ipykernel" in sys.modules
except Exception:
    _NB = False


def _fmt(v):
    if v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))):
        return ""
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else "no"
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        a = abs(float(v))
        if a >= 1000:
            return f"{v:,.0f}"
        if a >= 10:
            return f"{v:,.1f}"
        return f"{v:,.3f}".rstrip("0").rstrip(".") if a else "0"
    return str(v)


def _html(df):
    d = df.copy()
    for c in d.columns:
        d[c] = d[c].map(_fmt)
    return d.to_html(index=False, border=0, escape=True)


class _Cell:
    """Progress lines and result tables of one cell, rendered as a single output."""
    def start(self, title):
        self.title, self.lines, self.tables = title, [], []
        self.h = display(HTML(""), display_id=True) if _NB else None
        self._draw()

    def log(self, txt, replace_last=False):
        if replace_last and self.lines:
            self.lines[-1] = txt
        else:
            self.lines.append(txt)
        self._draw()
        if not _NB:
            print("   ", txt, flush=True)

    def show(self, *blocks):
        self.tables = list(blocks)
        self._draw()
        if not _NB:
            for lab, df in blocks:
                print(f"\n== {lab}")
                print(df.to_string(index=False, max_rows=80) if len(df) else "(empty)")

    def _draw(self):
        if getattr(self, "h", None) is None:
            return
        css = ("<style>.pkgc table{border-collapse:collapse;font-size:12px;margin:2px 0 12px}"
               ".pkgc th{background:#f1f0ec;color:#0b0b0b;text-align:left;padding:3px 8px;border-bottom:1px solid #b8b7b0}"
               ".pkgc td{padding:2px 8px;border-bottom:1px solid #e6e5e0}"
               ".pkgc h4{margin:10px 0 3px;font-size:13px}.pkgc .pg{color:#8a8985;font-size:12px}</style>")
        body = f"<div class='pkgc'><h3 style='margin:4px 0'>{self.title}</h3>"
        body += "".join(f"<div class='pg'>{x}</div>" for x in self.lines)
        body += "".join(f"<h4>{lab}</h4>{_html(df)}" for lab, df in self.tables)
        self.h.update(HTML(css + body + "</div>"))


CELL = _Cell()


@contextlib.contextmanager
def step(name):
    t0 = time.time()
    CELL.log(f"running: {name} (started {time.strftime('%H:%M')})")
    stop = threading.Event()

    def _tick():
        while not stop.wait(30):
            msg = f"running: {name} ({(time.time() - t0) / 60:.1f} min"
            try:
                st = spark.sparkContext.statusTracker()
                ids = st.getActiveStageIds()
                if ids:
                    info = st.getStageInfo(max(ids))
                    msg += f", stage {info.stageId}: {info.numCompletedTasks:,}/{info.numTasks:,} tasks"
            except Exception:
                pass
            if _NB:
                CELL.log(msg + ")", replace_last=True)

    th = threading.Thread(target=_tick, daemon=True)
    th.start()
    try:
        yield
    finally:
        stop.set()
        mins = (time.time() - t0) / 60
        if _NB:
            CELL.log(f"done: {name} ({mins:.1f} min)", replace_last=True)
        else:
            print(f"    done: {name} ({mins:.1f} min)", flush=True)


ANSWERS = []


def ans(q, a):
    ANSWERS.append(dict(question=q, answer=a))


def table_months(tbl):
    try:
        p = spark.sql(f"SHOW PARTITIONS {tbl}").toPandas().iloc[:, 0]
        m = sorted({x.split("month=")[1].split("/")[0] for x in p if "month=" in x})
        if m:
            return m
    except Exception:
        pass
    return sorted(r[0] for r in spark.table(tbl).select("month").distinct().collect())


def mi_of(m):
    return int(m[:4]) * 12 + int(m[5:7])


def m_of(i):
    return f"{(i - 1) // 12:04d}-{(i - 1) % 12 + 1:02d}"


mi_col = lambda c: (F.substring(F.col(c), 1, 4).cast("int") * 12 + F.substring(F.col(c), 6, 2).cast("int")).cast("int")
pct = lambda x, cols: x.assign(**{c: x[c].map(lambda v: "" if pd.isna(v) else f"{v:.1%}") for c in cols if c in x})
OUT_DIR = CONFIG["OUT_DIR"]
os.makedirs(OUT_DIR, exist_ok=True)


def cache_parquet(sdf, name):
    """write once, read back; reused on a rerun unless REBUILD"""
    sig = hashlib.md5(json.dumps({k: CONFIG[k] for k in ["PAIR_TABLE", "DIM_TABLE", "REG_STREAM", "ORG_PARTY_TYPES", "LAST_MONTH",
                                                         "FEATURE_START_AFTER", "SAMPLE_PCT"]}, default=str).encode()).hexdigest()[:8]
    path = f"{CONFIG['WORK_DIR']}/sig={sig}/{name}"
    if not CONFIG["REBUILD"]:
        try:
            out = spark.read.parquet(path)
            out.limit(1).count()
            CELL.log(f"reused work file: {name}")
            return out
        except Exception:
            pass
    sdf.write.mode("overwrite").parquet(path)
    return spark.read.parquet(path)


print("setup done")

## §1 · Inputs and sample

Population: TM organisations (power id not shared). A hash sample of `SAMPLE_PCT` % of them gives the streams that
are modelled; payer and customer measures (network inputs) use the whole population. Incoming streams from outside
PNC (`ext`) and from other PNC customers (`int`); own accounts at other banks and self transfers are left out.

In [ ]:
CELL.start("§1 · Inputs and sample")
with step("months, population, sample"):
    PM = table_months(CONFIG["PAIR_TABLE"])
    LAST = CONFIG["LAST_MONTH"] or PM[-1]
    PM = [m for m in PM if m <= LAST]
    M0 = PM[0]
    BASE = mi_of(M0)
    LAST_MI = mi_of(LAST) - BASE
    FEAT0 = CONFIG["FEATURE_START_AFTER"]            # first month with features (needs this much history)
    TRAIN_END_MI = mi_of(CONFIG["TRAIN_END"]) - BASE
    dim = spark.table(CONFIG["DIM_TABLE"])
    _dc = set(dim.columns)
    cond = F.lit(True)
    if "is_tm" in _dc:
        cond = cond & (F.col("is_tm").cast("boolean") == F.lit(True))
    if "party_type" in _dc:
        cond = cond & F.col("party_type").isin(CONFIG["ORG_PARTY_TYPES"])
    if "pwr_shared" in _dc:
        cond = cond & (F.coalesce(F.col("pwr_shared").cast("boolean"), F.lit(False)) == F.lit(False))
    POP = dim.where(cond).select(F.col("mdm_id").cast("string").alias("cust")).dropDuplicates().cache()
    SAMP = POP.where(F.abs(F.hash("cust")) % 100 < CONFIG["SAMPLE_PCT"]).cache()
    n_pop, n_samp = POP.count(), SAMP.count()
    _pc = set(spark.table(CONFIG["PAIR_TABLE"]).columns)
    # o_self is empty on rows from other PNC customers: a null must count as "not own account" (v1.0 dropped every int row)
    _own = (F.coalesce(F.lower(F.col("o_self").cast("string")).isin("true", "1", "y", "yes"), F.lit(False))
            if "o_self" in _pc else F.lit(False))
    # active incoming stream-months of the whole population
    A = (spark.table(CONFIG["PAIR_TABLE"])
         .where((F.col("dir") == "in") & F.col("scope").isin("int", "ext") & ~_own & F.col("month").between(M0, LAST))
         .select(F.col("cust").cast("string").alias("cust"), F.col("scope").alias("side"), F.col("other").cast("string").alias("other"),
                 (mi_col("month") - BASE).alias("mi"), F.col("amount").cast("double").alias("amount"))
         .join(POP, "cust", "left_semi")
         .groupBy("cust", "side", "other", "mi").agg(F.sum("amount").alias("amount"))
         .where(F.col("amount") > 0))
    A = cache_parquet(A, "active_v11")
CELL.show(("Inputs", pd.DataFrame([
    dict(item="months", value=f"{M0} .. {LAST} ({len(PM)})"),
    dict(item="features from", value=m_of(BASE + FEAT0)),
    dict(item="training months (as-of) up to", value=CONFIG["TRAIN_END"]),
    dict(item="test months (as-of)", value=f"{m_of(BASE + TRAIN_END_MI + 1)} .. {m_of(BASE + LAST_MI - 1)} (target = next month)"),
    dict(item="population", value=f"{n_pop:,} TM organisations"),
    dict(item="sample", value=f"{n_samp:,} ({CONFIG['SAMPLE_PCT']}%)")])))

## §2 · Stream-month panel

Every incoming stream of the sample, every month from its first payment to the end of the data, while it paid at
least once in the last 12 months. Features use data up to the month m; the targets are month m+1: does it pay
(`y_pay`) and how much (`y_amt`).

- **History:** paid this month, months paid out of the last 3 / 6 / 12, months since the last payment, months since
  the first, last amount, median of the last 3 paid amounts, average of the last 3 / 6 months (empty months = 0),
  variation of the paid amounts.
- **For BG/BB:** x = months paid after the first, t_x = last paid month − first, n = m − first (data start = first
  month for older relationships).
- **Network:** (1) the payer's PNC payees: share of those it paid last month that it did not pay this month, without
  this customer (5+ payees), and its payments to PNC payees this month against its 3-month average; (2) for a payer
  that is a PNC customer, its own income this month against its 6-month average (the shock upstream); (3) the
  customer's own income against its 6-month average, and the share of its other streams paid last month and not this
  month.

In [ ]:
CELL.start("§2 · Panel")


def build_grid(As, min_mi):
    """dense stream-month grid with history features and the next 3 months' amounts"""
    _first = As.groupBy("cust", "side", "other").agg(F.min("mi").alias("first"))
    G = (_first.withColumn("mi", F.explode(F.sequence(F.col("first"), F.lit(LAST_MI))))
         .join(As, ["cust", "side", "other", "mi"], "left").na.fill(0.0, ["amount"])
         .withColumn("act", (F.col("amount") > 0).cast("int")))
    _w = W.partitionBy("cust", "side", "other").orderBy("mi")
    _rw = lambda a, b: _w.rowsBetween(a, b)
    G = (G.withColumn("n3", F.sum("act").over(_rw(-2, 0))).withColumn("n6", F.sum("act").over(_rw(-5, 0)))
         .withColumn("n12", F.sum("act").over(_rw(-11, 0)))
         .withColumn("s3", F.sum("amount").over(_rw(-2, 0))).withColumn("s6", F.sum("amount").over(_rw(-5, 0)))
         .withColumn("s12", F.sum("amount").over(_rw(-11, 0)))
         .withColumn("q12", F.sum(F.col("amount") * F.col("amount")).over(_rw(-11, 0)))
         .withColumn("last_mi", F.max(F.when(F.col("act") == 1, F.col("mi"))).over(_rw(W.unboundedPreceding, 0)))
         .withColumn("x", F.sum("act").over(_rw(W.unboundedPreceding, 0)) - 1)
         .withColumn("last_amt", F.last(F.when(F.col("act") == 1, F.col("amount")), ignorenulls=True).over(_rw(W.unboundedPreceding, 0)))
         .withColumn("act_prev", F.lag("act", 1, 0).over(_w))
         .withColumn("y_pay", F.lead("act", 1).over(_w)).withColumn("y_amt", F.lead("amount", 1).over(_w))
         .withColumn("y_amt2", F.lead("amount", 2).over(_w)).withColumn("y_amt3", F.lead("amount", 3).over(_w)))
    # median of the last 3 paid amounts, carried forward over empty months
    _pa = (As.withColumn("l3", F.collect_list("amount").over(W.partitionBy("cust", "side", "other").orderBy("mi").rowsBetween(-2, 0)))
           .select("cust", "side", "other", "mi", F.expr("element_at(array_sort(l3), cast(ceil(size(l3) / 2.0) as int))").alias("med3p")))
    G = (G.join(_pa, ["cust", "side", "other", "mi"], "left")
         .withColumn("med3p", F.last("med3p", ignorenulls=True).over(_rw(W.unboundedPreceding, 0)))
         .withColumn("rec", F.col("mi") - F.col("last_mi")).withColumn("n", F.col("mi") - F.col("first"))
         .withColumn("t_x", F.col("last_mi") - F.col("first"))
         .withColumn("mean_p", F.col("s12") / F.col("n12"))
         .withColumn("cv_p", F.sqrt(F.greatest(F.col("q12") / F.col("n12") - F.col("mean_p") * F.col("mean_p"), F.lit(0.0))) / F.col("mean_p"))
         .where((F.col("rec") <= 11) & F.col("y_pay").isNotNull() & (F.col("mi") >= min_mi)))
    return G


with step("dense stream-month grid of the sample"):
    G = build_grid(A.join(SAMP, "cust", "left_semi"), FEAT0)

with step("network inputs from the whole population"):
    _pay = G.select("side", "other").distinct()
    Ap = A.join(_pay, ["side", "other"], "left_semi")
    _prev = Ap.select("cust", "side", "other", (F.col("mi") + 1).alias("mi"), F.lit(1).alias("p"))
    PS = (Ap.select("cust", "side", "other", "mi", F.lit(1).alias("c")).join(_prev, ["cust", "side", "other", "mi"], "full")
          .groupBy("side", "other", "mi").agg(F.sum(F.coalesce(F.col("p"), F.lit(0))).alias("pn_prev"),
                                              F.sum(F.when((F.col("p") == 1) & F.col("c").isNull(), 1).otherwise(0)).alias("pn_miss")))
    _pt = Ap.groupBy("side", "other", "mi").agg(F.sum("amount").alias("p_tot"))
    _wp = W.partitionBy("side", "other").orderBy("mi")
    _pt = (_pt.withColumn("p_avg3", F.avg("p_tot").over(_wp.rangeBetween(-3, -1)))
           .select("side", "other", "mi", F.log((F.col("p_tot") + 1) / (F.col("p_avg3") + 1)).alias("payer_out_ratio")))
    PS = PS.join(_pt, ["side", "other", "mi"], "left")
    # income ratio of every population customer (the customer itself, and payers that are PNC customers)
    _ti = A.groupBy("cust", "mi").agg(F.sum("amount").alias("tin"))
    _ti = (_ti.withColumn("avg6", F.avg("tin").over(W.partitionBy("cust").orderBy("mi").rangeBetween(-6, -1)))
           .select("cust", "mi", F.log((F.col("tin") + 1) / (F.coalesce(F.col("avg6"), F.lit(0.0)) + 1)).alias("in_ratio")))
    _cm = (G.groupBy("cust", "mi").agg(F.sum("act_prev").alias("c_prev"),
                                       F.sum(((F.col("act_prev") == 1) & (F.col("act") == 0)).cast("int")).alias("c_miss")))
    P = (G.join(PS, ["side", "other", "mi"], "left").na.fill(0, ["pn_prev", "pn_miss"])
         .withColumn("o_prev", F.col("pn_prev") - F.col("act_prev"))
         .withColumn("payer_miss", F.when(F.col("o_prev") >= 5, (F.col("pn_miss") - ((F.col("act_prev") == 1) & (F.col("act") == 0)).cast("int"))
                                          / F.col("o_prev")))
         .join(_ti.withColumnRenamed("in_ratio", "cust_in_ratio"), ["cust", "mi"], "left")
         .join(_ti.select(F.col("cust").alias("other"), "mi", F.col("in_ratio").alias("payer_in_ratio")).withColumn("side", F.lit("int")),
               ["side", "other", "mi"], "left")
         .join(_cm, ["cust", "mi"], "left")
         .withColumn("cust_miss", F.when(F.col("c_prev") - F.col("act_prev") >= 2,
                                         (F.col("c_miss") - ((F.col("act_prev") == 1) & (F.col("act") == 0)).cast("int"))
                                         / (F.col("c_prev") - F.col("act_prev")))))
    if CONFIG["REG_STREAM"]:
        try:
            _rs = spark.table(CONFIG["REG_STREAM"])
            _rc = set(_rs.columns)
            _pcn = F.col("p_continue").cast("double") if "p_continue" in _rc else F.lit(1.0)
            _f1n = F.col("fc1_n").cast("double") if "fc1_n" in _rc else F.lit(1.0)
            _f1a = F.col("fc1_amt").cast("double") if "fc1_amt" in _rc else F.lit(None).cast("double")
            _rsx = (_rs.where((F.col("dir") == "in") & F.col("side").isin("int", "ext") & F.col("cadence").isNotNull())
                    .select(F.col("cust").cast("string").alias("cust"), F.col("side").cast("string").alias("side"),
                            F.col("other").cast("string").alias("other"), (mi_col("month") - BASE).alias("mi"),
                            F.when(_f1n > 0, F.coalesce(_pcn, F.lit(1.0))).otherwise(F.lit(0.0)).alias("per_p"),
                            (_f1a * F.coalesce(_pcn, F.lit(1.0))).alias("per_d"))
                    .join(SAMP, "cust", "left_semi")
                    .groupBy("cust", "side", "other", "mi").agg(F.max("per_p").alias("per_p"), F.max("per_d").alias("per_d")))
            P = P.join(_rsx, ["cust", "side", "other", "mi"], "left")
            HAVE_PER = True
        except Exception as ex_:
            P = P.withColumn("per_p", F.lit(None).cast("double")).withColumn("per_d", F.lit(None).cast("double"))
            HAVE_PER = False
            CELL.log(f"periodicity not used: {str(ex_)[:150]}")
    else:
        P = P.withColumn("per_p", F.lit(None).cast("double")).withColumn("per_d", F.lit(None).cast("double"))
        HAVE_PER = False
    KEEP = ["cust", "side", "other", "mi", "act", "n3", "n6", "n12", "s3", "s6", "rec", "n", "x", "t_x", "last_amt", "med3p",
            "mean_p", "cv_p", "payer_miss", "payer_out_ratio", "payer_in_ratio", "cust_in_ratio", "cust_miss", "per_p", "per_d",
            "y_pay", "y_amt", "y_amt2", "y_amt3"]
    PANEL = cache_parquet(P.select(*KEEP), f"panel_v11_s{CONFIG['SAMPLE_PCT']}")
    _np = PANEL.groupBy((F.col("mi") <= TRAIN_END_MI).alias("train")).agg(F.count(F.lit(1)).alias("rows"),
                                                                          F.avg("y_pay").alias("pays_next")).toPandas()
CELL.show(("Panel rows", _np.assign(train=np.where(_np.train, "training months", "test months"))))

## §3 · BG/BB: is the relationship still alive?

Discrete-time buy-till-you-die (Fader, Hardie and Shang 2010). Each month a live relationship pays with its own
probability p (Beta(α, β) across relationships) and may end for good with probability θ (Beta(γ, δ)). Fitted by
maximum likelihood on every stream's history as of the last training month (x, t_x, n are all it needs), separately
per segment (PNC or outside payer × usual paid amount band). Then for any stream-month:
P(pays next month) = P(alive next month | x, t_x, n) × (α + x) / (α + β + n), and expected dollars = that × the median
of its last 3 paid amounts.

In [ ]:
CELL.start("§3 · BG/BB")
try:
    from scipy.special import betaln as _betaln
except Exception:
    _lg = np.vectorize(math.lgamma)
    _betaln = lambda a, b: _lg(a) + _lg(b) - _lg(a + b)
try:
    from scipy.optimize import minimize as _minimize
except Exception:
    _minimize = None


def _nm(f, x0, iters=600):
    """small Nelder-Mead, used only if scipy is missing"""
    n = len(x0)
    S = [np.array(x0, float)] + [np.array(x0, float) + np.eye(n)[i] * 0.5 for i in range(n)]
    V = [f(s) for s in S]
    for _ in range(iters):
        o = np.argsort(V); S = [S[i] for i in o]; V = [V[i] for i in o]
        c = np.mean(S[:-1], axis=0)
        r = c + (c - S[-1]); fr = f(r)
        if fr < V[0]:
            e = c + 2 * (c - S[-1]); fe = f(e)
            S[-1], V[-1] = (e, fe) if fe < fr else (r, fr)
        elif fr < V[-2]:
            S[-1], V[-1] = r, fr
        else:
            k = c + 0.5 * (S[-1] - c); fk = f(k)
            if fk < V[-1]:
                S[-1], V[-1] = k, fk
            else:
                S = [S[0] + 0.5 * (s - S[0]) for s in S]; V = [f(s) for s in S]
    return S[int(np.argmin(V))]


def _bgbb_ll(par, x, tx, n):
    a, b, g, d = np.exp(par)
    base_ab, base_gd = _betaln(a, b), _betaln(g, d)
    t1 = _betaln(a + x, b + n - x) - base_ab + _betaln(g, d + n) - base_gd
    terms = [t1]
    imax = int(np.max(n - tx)) if len(n) else 0
    for i in range(imax):
        ok = i <= n - tx - 1
        ti = _betaln(a + x, b + tx - x + i) - base_ab + _betaln(g + 1, d + tx + i) - base_gd
        terms.append(np.where(ok, ti, -np.inf))
    return np.logaddexp.reduce(np.vstack(terms), axis=0)


def bgbb_fit(df):
    x, tx, n, w = (df[c].to_numpy(float) for c in ["x", "t_x", "n", "w"])
    f = lambda p: -np.sum(w * _bgbb_ll(p, x, tx, n)) / w.sum()
    x0 = np.log([1.0, 1.0, 1.0, 5.0])
    if _minimize is not None:
        r = _minimize(f, x0, method="L-BFGS-B", bounds=[(-6, 6)] * 4)
        return np.exp(r.x)
    return np.exp(_nm(f, x0))


def bgbb_pay(par, x, tx, n):
    a, b, g, d = par
    ll = _bgbb_ll(np.log(par), x, tx, n)
    alive = np.exp(_betaln(a + x, b + n - x) - _betaln(a, b) + _betaln(g, d + n + 1) - _betaln(g, d) - ll)
    return np.clip(alive, 0, 1) * (a + x) / (a + b + n)


def bgbb_k(par, x, tx, n, K=3):
    """P(pays) in each of the next K months: P(alive at n+k | data) x posterior mean of p"""
    a, b, g, d = par
    ll = _bgbb_ll(np.log(par), x, tx, n)
    base = _betaln(a + x, b + n - x) - _betaln(a, b) - _betaln(g, d) - ll
    mp = (a + x) / (a + b + n)
    return [np.clip(np.exp(base + _betaln(g, d + n + k)), 0, 1) * mp for k in range(1, K + 1)]


def bgbb_lookup(nmax):
    """every (segment, x, t_x, n) the data can hold: P(pays next month), expected paying months in the next 3, and their
    variance (sum of p(1-p)); joined to Spark tables instead of computing BG/BB there"""
    rows = [(x, tx, n) for n in range(nmax + 1) for x in range(n + 1) for tx in ([0] if x == 0 else range(x, n + 1))]
    base = pd.DataFrame(rows, columns=["x", "t_x", "n"])
    out = []
    for sg, par in BG.items():
        pk = bgbb_k(par, base.x.to_numpy(float), base.t_x.to_numpy(float), base.n.to_numpy(float))
        out.append(base.assign(seg=sg, p1=pk[0], e3=sum(pk), vc3=sum(q * (1 - q) for q in pk)))
    return pd.concat(out, ignore_index=True)


AMT_BANDS = CONFIG["AMOUNT_BANDS"]
_band = lambda c: (F.when(F.col(c) < AMT_BANDS[0], "under 10K").when(F.col(c) < AMT_BANDS[1], "10K-100K")
                   .when(F.col(c) < AMT_BANDS[2], "100K-1M").otherwise("1M+"))
with step("histories as of the last training month, grouped by (segment, x, t_x, n)"):
    _h = (PANEL.where(F.col("mi") == TRAIN_END_MI)
          .unionByName(  # streams that had stopped (12+ months silent) are not in the panel; rebuild them from A
              A.join(SAMP, "cust", "left_semi").where(F.col("mi") <= TRAIN_END_MI)
              .groupBy("cust", "side", "other").agg(F.min("mi").alias("first"), F.max("mi").alias("last_mi"),
                                                    F.count(F.lit(1)).alias("cnt"), F.expr("percentile_approx(amount, 0.5)").alias("med3p"))
              .where(TRAIN_END_MI - F.col("last_mi") > 11)
              .select("cust", "side", "other", F.lit(TRAIN_END_MI).alias("mi"), (F.col("cnt") - 1).alias("x"),
                      (F.col("last_mi") - F.col("first")).alias("t_x"), (F.lit(TRAIN_END_MI) - F.col("first")).alias("n"), "med3p"),
              allowMissingColumns=True)
          .withColumn("seg", F.concat(F.col("side"), F.lit(" · "), _band("med3p")))
          .groupBy("seg", "x", "t_x", "n").agg(F.count(F.lit(1)).alias("w")).toPandas())
with step("fit per segment"):
    BG = {}
    rows = []
    for sg, d in _h.groupby("seg"):
        if d.w.sum() < CONFIG["BGBB_MIN_STREAMS"]:
            continue
        par = bgbb_fit(d)
        BG[sg] = par
        a, b, g, dd = par
        rows.append(dict(segment=sg, streams=int(d.w.sum()), alpha=a, beta=b, gamma=g, delta=dd,
                         mean_monthly_pay_prob=a / (a + b), mean_monthly_end_prob=g / (g + dd)))
    _all = bgbb_fit(_h.groupby(["x", "t_x", "n"], as_index=False).w.sum())
    BG["all"] = _all
    rows.append(dict(segment="all", streams=int(_h.w.sum()), alpha=_all[0], beta=_all[1], gamma=_all[2], delta=_all[3],
                     mean_monthly_pay_prob=_all[0] / (_all[0] + _all[1]), mean_monthly_end_prob=_all[2] / (_all[2] + _all[3])))
    bg_tab = pd.DataFrame(rows)
CELL.show(("BG/BB parameters by segment (pay = chance a live relationship pays in a month; end = chance it ends in a month)",
           pct(bg_tab, ["mean_monthly_pay_prob", "mean_monthly_end_prob"])))

## §4 · Predicting next month, stream by stream

A sample of training rows fits the logistic models; test rows (as-of months after the training months) score every
model. Expected dollars = P(pays) × the median of the last 3 paid amounts for the probability models.

| model | P(pays next month) | expected dollars |
|---|---|---|
| last month | paid this month (1 / 0) | this month's amount |
| 3-month average | months paid in the last 3 ÷ 3 | average of the last 3 months |
| steady rule | 1 if paid this month, ≥ 4 of 6, ≥ 8 of 12; else months paid of 6 ÷ 6 | median of last 3 paid if steady, else 6-month average |
| periodicity | scheduled streams: chance to continue; others: 3-month average | `fc1_amt` × chance to continue; others: 3-month average |
| BG/BB | §3 | P × median of last 3 paid |
| history (logistic) | months paid, recency, tenure, amounts, variation, side | P × median of last 3 paid |
| history + BG/BB | history + logit of the BG/BB probability | same |
| history + BG/BB + network | + payer misses, payer's PNC payments, upstream shock, own income, own other streams | same |

Scores: AUC, log loss and calibration of P; dollar error per stream and per customer (sum of its incoming streams
against what came in from them), as weighted absolute % error (WAPE). Also AUC by history class, to see which model
generalises to irregular relationships.

In [ ]:
CELL.start("§4 · Next-month prediction")
with step("training and test samples to pandas"):
    _ntr = PANEL.where(F.col("mi") <= TRAIN_END_MI).count()
    _nte = PANEL.where(F.col("mi") > TRAIN_END_MI).count()
    TR = (PANEL.where(F.col("mi") <= TRAIN_END_MI)
          .where(F.rand(1) < min(1.0, CONFIG["TRAIN_ROWS"] / max(_ntr, 1))).toPandas())
    TE = (PANEL.where(F.col("mi") > TRAIN_END_MI)
          .where(F.abs(F.hash("cust")) % 1000 < int(1000 * min(1.0, CONFIG["TEST_ROWS"] / max(_nte, 1)))).toPandas())


def _seg(d):
    v = d.med3p.fillna(0).to_numpy()
    b = np.where(v < AMT_BANDS[0], "under 10K", np.where(v < AMT_BANDS[1], "10K-100K", np.where(v < AMT_BANDS[2], "100K-1M", "1M+")))
    return d.side.astype(str).to_numpy() + " · " + b


def _hist_class(d):
    steady = (d.act == 1) & (d.n6 >= 4) & (d.n12 >= 8)
    return np.select([steady, d.n12 >= 8, d.n12 >= 3], ["steady", "frequent, uneven", "occasional"], "rare")


def add_preds(d):
    d = d.copy()
    amt = d.med3p.fillna(0)
    d["p_last"] = d.act.astype(float)
    d["d_last"] = np.where(d.act == 1, d.last_amt.fillna(0), 0.0)
    d["p_avg3"], d["d_avg3"] = d.n3 / 3.0, d.s3 / 3.0
    st = (d.act == 1) & (d.n6 >= 4) & (d.n12 >= 8)
    d["p_steady"] = np.where(st, 1.0, d.n6 / 6.0)
    d["d_steady"] = np.where(st, amt, d.s6 / 6.0)
    if HAVE_PER:
        has = d.per_p.notna()
        d["p_period"] = np.where(has, d.per_p, d.p_avg3)
        d["d_period"] = np.where(has & d.per_d.notna(), d.per_d, d.d_avg3)
    seg = _seg(d)
    p = np.full(len(d), np.nan)
    for sg in np.unique(seg):
        m = seg == sg
        par = BG.get(sg, BG["all"])
        p[m] = bgbb_pay(par, d.x.to_numpy(float)[m], d.t_x.to_numpy(float)[m], d.n.to_numpy(float)[m])
    d["p_bgbb"] = np.clip(p, 1e-6, 1 - 1e-6)
    d["d_bgbb"] = d.p_bgbb * amt
    d["hist_class"] = _hist_class(d)
    return d


HIST_F = ["act", "n3", "n6", "n12", "rec", "log_n", "log_last", "log_med3", "cv_p", "is_int", "single_payment"]
NET_F = ["payer_miss", "payer_out_ratio", "payer_in_ratio", "cust_in_ratio", "cust_miss"]


def features(d, cols):
    X = pd.DataFrame(index=d.index)
    X["act"], X["n3"], X["n6"], X["n12"] = d.act, d.n3, d.n6, d.n12
    X["rec"] = d.rec.clip(0, 12)
    X["log_n"] = np.log1p(d.n.clip(lower=0))
    X["log_last"] = np.log1p(d.last_amt.fillna(0).clip(lower=0))
    X["log_med3"] = np.log1p(d.med3p.fillna(0).clip(lower=0))
    X["cv_p"] = d.cv_p.fillna(0).clip(0, 5)
    X["is_int"] = (d.side == "int").astype(float)
    X["single_payment"] = (d.x == 0).astype(float)
    X["bgbb_logit"] = np.log(d.p_bgbb / (1 - d.p_bgbb))
    for c in NET_F:
        v = d[c].astype(float)
        X[c] = v.fillna(0).clip(-5, 5)
        X[c + "_na"] = v.isna().astype(float)
    use = []
    for c in cols:
        use += [c] + ([c + "_na"] if c in NET_F else [])
    return X[use].to_numpy(float), use


def logit_fit(X, y, lam=1e-3, iters=40):
    mu_, sd_ = X.mean(0), X.std(0)
    sd_[sd_ == 0] = 1
    Z = np.column_stack([np.ones(len(X)), (X - mu_) / sd_])
    b = np.zeros(Z.shape[1])
    for _ in range(iters):
        p = 1 / (1 + np.exp(-np.clip(Z @ b, -30, 30)))
        wgt = p * (1 - p)
        H = (Z * wgt[:, None]).T @ Z + lam * np.eye(len(b))
        st = np.linalg.solve(H, Z.T @ (y - p) - lam * b)
        b += st
        if np.max(np.abs(st)) < 1e-7:
            break
    return dict(b=b, mu=mu_, sd=sd_)


def logit_pred(m, X):
    Z = np.column_stack([np.ones(len(X)), (X - m["mu"]) / m["sd"]])
    return np.clip(1 / (1 + np.exp(-np.clip(Z @ m["b"], -30, 30))), 1e-6, 1 - 1e-6)


def auc(s, y):
    r = pd.Series(s).rank().to_numpy()
    n1 = y.sum(); n0 = len(y) - n1
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / max(n1 * n0, 1)) if n1 and n0 else np.nan


def logloss(p, y):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


with step("BG/BB and simple predictions"):
    TR, TE = add_preds(TR), add_preds(TE)
with step("logistic models"):
    LM = {}
    for nm_, cols in [("history", HIST_F), ("history + BG/BB", HIST_F + ["bgbb_logit"]),
                      ("history + BG/BB + network", HIST_F + ["bgbb_logit"] + NET_F)]:
        Xtr, use = features(TR, cols)
        LM[nm_] = (logit_fit(Xtr, TR.y_pay.to_numpy(float)), cols, use)
        Xte, _ = features(TE, cols)
        TE["p_" + nm_] = logit_pred(LM[nm_][0], Xte)
        TE["d_" + nm_] = TE["p_" + nm_] * TE.med3p.fillna(0)

# --- amount model: log of next month's amount when it pays -------------------------------------------------
_cl = lambda c, lo, hi: F.least(F.greatest(F.coalesce(F.col(c).cast("double"), F.lit(0.0)), F.lit(lo)), F.lit(hi))
AMT_SPEC = [  # name, pandas, spark
    ("log_last", lambda d: np.log1p(d.last_amt.fillna(0).clip(lower=0)), F.log1p(_cl("last_amt", 0, 1e15))),
    ("log_med3", lambda d: np.log1p(d.med3p.fillna(0).clip(lower=0)), F.log1p(_cl("med3p", 0, 1e15))),
    ("log_mean", lambda d: np.log1p(d.mean_p.fillna(0).clip(lower=0)), F.log1p(_cl("mean_p", 0, 1e15))),
    ("log_avg3", lambda d: np.log1p((d.s3 / 3).clip(lower=0)), F.log1p(_cl("s3", 0, 1e15) / 3)),
    ("log_avg6", lambda d: np.log1p((d.s6 / 6).clip(lower=0)), F.log1p(_cl("s6", 0, 1e15) / 6)),
    ("cv", lambda d: d.cv_p.fillna(0).clip(0, 5), _cl("cv_p", 0, 5)),
    ("n12", lambda d: d.n12.astype(float), F.col("n12").cast("double")),
    ("rec", lambda d: d.rec.clip(0, 12).astype(float), _cl("rec", 0, 12)),
    ("act", lambda d: d.act.astype(float), F.col("act").cast("double")),
    ("is_int", lambda d: (d.side == "int").astype(float), (F.col("side") == "int").cast("double")),
    ("log_n", lambda d: np.log1p(d.n.clip(lower=0)), F.log1p(_cl("n", 0, 1e6))),
]


def amt_fit(d):
    X = np.column_stack([f(d) for _, f, _ in AMT_SPEC]).astype(float)
    yv = np.log1p(d.y_amt.to_numpy(float))
    mu_, sd_ = X.mean(0), X.std(0)
    sd_[sd_ == 0] = 1
    Z = np.column_stack([np.ones(len(X)), (X - mu_) / sd_])
    b = np.linalg.solve(Z.T @ Z + 1e-3 * np.eye(Z.shape[1]), Z.T @ yv)
    r = yv - Z @ b
    return dict(b=b, mu=mu_, sd=sd_, s2=float(np.var(r)), smear=float(np.mean(np.exp(r))))


def amt_pred(m, d):
    X = np.column_stack([f(d) for _, f, _ in AMT_SPEC]).astype(float)
    Z = np.column_stack([np.ones(len(X)), (X - m["mu"]) / m["sd"]])
    return np.clip(np.exp(Z @ m["b"]) * m["smear"] - 1, 0, None)


def amt_spark(m):
    e = F.lit(float(m["b"][0]))
    for j, (_, _, sx) in enumerate(AMT_SPEC):
        e = e + F.lit(float(m["b"][j + 1])) * (sx - F.lit(float(m["mu"][j]))) / F.lit(float(m["sd"][j]))
    return F.greatest(F.exp(e) * F.lit(m["smear"]) - 1, F.lit(0.0))


with step("amount model (next month's amount when it pays)"):
    AM = amt_fit(TR[TR.y_pay == 1])
    for d_ in (TR, TE):
        d_["amt_model"] = amt_pred(AM, d_)
    TE["d_bgbb_am"] = TE.p_bgbb * TE.amt_model
    TE["p_bgbb_am"] = TE.p_bgbb
    TE["d_full_am"] = TE["p_history + BG/BB + network"] * TE.amt_model
    TE["p_full_am"] = TE["p_history + BG/BB + network"]
    _pp = TE[TE.y_pay == 1]
    _ar = lambda lab, v: dict(amount=lab, wape=float(np.abs(v - _pp.y_amt).sum() / _pp.y_amt.sum()),
                              within_10pct=float((np.abs(v - _pp.y_amt) <= 0.1 * _pp.y_amt).mean()))
    AMT_TAB = pd.DataFrame([_ar("median of last 3 paid", _pp.med3p.fillna(0)), _ar("last paid amount", _pp.last_amt.fillna(0)),
                            _ar("amount model", _pp.amt_model)])

MODELS = [("last month", "last"), ("3-month average", "avg3"), ("steady rule", "steady")] + \
         ([("periodicity, else 3-month average", "period")] if HAVE_PER else []) + \
         [("BG/BB", "bgbb"), ("history (logistic)", "history"), ("history + BG/BB", "history + BG/BB"),
          ("history + BG/BB + network", "history + BG/BB + network"),
          ("BG/BB × amount model", "bgbb_am"), ("history + BG/BB + network × amount model", "full_am")]
with step("scores"):
    y = TE.y_pay.to_numpy(float)
    ya = TE.y_amt.fillna(0).to_numpy(float)
    rows, cls_rows = [], []
    _cg = TE.groupby(["cust", "mi"]).ngroup().to_numpy()
    _cy = np.bincount(_cg, weights=ya)
    for lab, k in MODELS:
        p, dd = TE[f"p_{k}"].to_numpy(float), TE[f"d_{k}"].to_numpy(float)
        cd = np.bincount(_cg, weights=dd)
        ok = _cy > 0
        rows.append(dict(model=lab, auc=auc(p, y), log_loss=logloss(p, y), mean_p=float(p.mean()), pays_next=float(y.mean()),
                         stream_wape=float(np.abs(dd - ya).sum() / ya.sum()),
                         customer_wape=float(np.abs(cd - _cy).sum() / _cy.sum()),
                         customer_median_abs_err=float(np.median(np.abs(cd[ok] - _cy[ok]) / _cy[ok]))))
        for hc in ["steady", "frequent, uneven", "occasional", "rare"]:
            m = (TE.hist_class == hc).to_numpy()
            if m.sum() > 200 and 0 < y[m].mean() < 1:
                cls_rows.append(dict(model=lab, history_class=hc, rows=int(m.sum()), auc=auc(p[m], y[m]),
                                     log_loss=logloss(p[m], y[m]), wape=float(np.abs(dd[m] - ya[m]).sum() / max(ya[m].sum(), 1))))
    SC = pd.DataFrame(rows)
    SCC = pd.DataFrame(cls_rows)
    SCW = SCC.pivot_table(index="model", columns="history_class", values="auc").reindex([m[0] for m in MODELS]).reset_index()
    # calibration of the best probability model, by decile of P
    best = SC[~SC.model.str.contains("amount model")].sort_values("log_loss").model.iloc[0]
    _bk = dict(MODELS)[best]
    _q = pd.qcut(TE[f"p_{_bk}"].rank(method="first"), 10, labels=False)
    CAL = (TE.assign(q=_q).groupby("q").agg(mean_p=(f"p_{_bk}", "mean"), pays_next=("y_pay", "mean"), rows=("y_pay", "size"))
           .reset_index().rename(columns={"q": "decile"}))
    # network coefficients (standardised)
    _m, _cols, _use = LM["history + BG/BB + network"]
    COEF = pd.DataFrame(dict(feature=["intercept"] + _use, coef_per_sd=_m["b"]))
    _Xn, _ = features(TE, HIST_F + ["bgbb_logit"] + NET_F)
    _na = {u: float(1 - np.mean(_Xn[:, i])) for i, u in enumerate(_use) if u.endswith("_na")}
    COEF["present_share"] = COEF.feature.map(lambda f_: _na.get(f_ + "_na", np.nan))
    COEF = COEF[COEF.feature.isin(NET_F + [c + "_na" for c in NET_F] + ["bgbb_logit", "act", "n12", "rec"])]
for _n, _d in [("amount_rules", AMT_TAB), ("scores", SC), ("scores_by_class", SCC), ("calibration", CAL), ("network_coefficients", COEF), ("bgbb_parameters", bg_tab)]:
    _d.to_csv(os.path.join(OUT_DIR, f"esb_{_n}.csv"), index=False)
CELL.show(("Test months: P(pays next month) and expected dollars, by model",
           pct(SC.assign(auc=SC.auc.round(4), log_loss=SC.log_loss.round(4)),
               ["mean_p", "pays_next", "stream_wape", "customer_wape", "customer_median_abs_err"])),
          ("AUC by history class at month m (steady / frequent, uneven / occasional / rare)", SCW.round(4)),
          ("Next month's amount when it pays: error by amount rule (test months)", pct(AMT_TAB, ["wape", "within_10pct"])),
          (f"Calibration of the best model by log loss ({best}): deciles of P", pct(CAL, ["mean_p", "pays_next"])),
          ("Network and BG/BB inputs in the full model (logistic coefficient per standard deviation)", COEF.round(3)))
_b = SC.set_index("model")
ans("Best model by log loss / by customer dollar error", f"{best} / {SC.sort_values('customer_wape').model.iloc[0]}")
for k in ["steady rule", "BG/BB", "history (logistic)", "history + BG/BB + network"]:
    if k in _b.index:
        ans(f"{k}: AUC / log loss / customer WAPE", f"{_b.loc[k, 'auc']:.3f} / {_b.loc[k, 'log_loss']:.4f} / {_b.loc[k, 'customer_wape']:.1%}")
if {"history + BG/BB", "history + BG/BB + network"} <= set(_b.index):
    ans("Network lift: AUC / log loss change from adding network inputs",
        f"{_b.loc['history + BG/BB + network', 'auc'] - _b.loc['history + BG/BB', 'auc']:+.4f} / "
        f"{_b.loc['history + BG/BB + network', 'log_loss'] - _b.loc['history + BG/BB', 'log_loss']:+.4f}")

## §5 · Surprise over 3 months: the candidate event

As of month m, for each customer: expected incoming dollars over m+1..m+3 from its relationships (BG/BB expected
paying months × the amount model) against what came from them. The same window the current rule uses (e−1..e+1
with e = m+2).

- **Surprise** = (came − expected) ÷ 3 months of usual inflow from these relationships.
- **z** = (came − expected) ÷ its standard deviation (whether it pays: sum of p(1−p) per month; how much: the amount
  model's spread). A z of −2 or less is unlikely to be ordinary noise.
- A weak relationship (low P) carries little expected money, so its silence hardly counts: the surprise weights every
  relationship by its strength, as the definition grid suggested.
- **Current-like event**: steady relationships (paid this month, ≥ 4 of 6, ≥ 8 of 12) paying less than half their
  usual over the 3 months, worth ≥ 20% of usual inflow.

Relationships that start in the window are not in the panel, so this is about losses on existing relationships.

In [ ]:
CELL.start("§5 · Surprise over 3 months")
with step("3-month expectations and surprise in the test months"):
    T3 = TE[TE.y_amt3.notna()].copy()
    _seg3 = _seg(T3)
    E3, VC3 = np.zeros(len(T3)), np.zeros(len(T3))
    for sg in np.unique(_seg3):
        m_ = _seg3 == sg
        pk = bgbb_k(BG.get(sg, BG["all"]), T3.x.to_numpy(float)[m_], T3.t_x.to_numpy(float)[m_], T3.n.to_numpy(float)[m_])
        E3[m_], VC3[m_] = sum(pk), sum(q * (1 - q) for q in pk)
    _ai = T3.amt_model.to_numpy(float)
    T3["exp3"] = E3 * _ai
    T3["var3"] = _ai ** 2 * (VC3 + (np.exp(AM["s2"]) - 1) * E3)
    T3["act3"] = T3.y_amt.fillna(0) + T3.y_amt2.fillna(0) + T3.y_amt3.fillna(0)
    T3["usual"] = T3.s6 / 6.0
    _st = (T3.act == 1) & (T3.n6 >= 4) & (T3.n12 >= 8)
    T3["steady_lost"] = np.where(_st & (T3.act3 <= 1.5 * T3.med3p.fillna(0)), np.maximum(3 * T3.med3p.fillna(0) - T3.act3, 0), 0.0)
    T3["neg"] = np.minimum(T3.act3 - T3.exp3, 0)
    CS = (T3.groupby(["cust", "mi"]).agg(exp3=("exp3", "sum"), act3=("act3", "sum"), var3=("var3", "sum"), usual=("usual", "sum"),
                                         steady_lost=("steady_lost", "sum")).reset_index())
    CS = CS[CS.usual > 0]
    CS["surprise"] = (CS.act3 - CS.exp3) / (3 * CS.usual)
    CS["z"] = (CS.act3 - CS.exp3) / np.sqrt(CS.var3.clip(lower=1e-9))
    CS["surprise_event"] = (CS.surprise <= -0.2) & (CS.z <= -2)
    CS["current_like"] = CS.steady_lost >= 0.2 * 3 * CS.usual
    bins = [-np.inf, -0.5, -0.2, -0.1, 0.1, 0.2, 0.5, np.inf]
    labs = ["-50% or worse", "-50% to -20%", "-20% to -10%", "-10% to +10%", "+10% to +20%", "+20% to +50%", "+50% or more"]
    CS["band"] = pd.cut(CS.surprise, bins, labels=labs)
    SUR = (CS.groupby("band", observed=False).agg(customer_months=("cust", "size"), z_le_minus2=("z", lambda v: float((v <= -2).mean())),
                                                  median_z=("z", "median"), current_like_event=("current_like", "mean")).reset_index())
    SUR["share"] = SUR.customer_months / SUR.customer_months.sum()
    OVL = pd.crosstab(np.where(CS.surprise_event, "surprise event", "no surprise event"),
                      np.where(CS.current_like, "current-like event", "no current-like event")).reset_index().rename(columns={"row_0": ""})
    # where the shortfall of surprise events comes from
    _ev = CS[CS.surprise_event][["cust", "mi"]]
    _src = T3.merge(_ev, on=["cust", "mi"]).groupby("hist_class").neg.sum()
    SRC = (-_src / -_src.sum()).rename("share_of_shortfall").reset_index() if len(_src) and _src.sum() < 0 else pd.DataFrame()
    for _n, _d in [("surprise_bands", SUR), ("surprise_overlap", OVL), ("surprise_sources", SRC)]:
        _d.to_csv(os.path.join(OUT_DIR, f"esb_{_n}.csv"), index=False)
CELL.show(("Customer-months by 3-month surprise (came − expected, share of 3 months of usual inflow)",
           pct(SUR, ["share", "z_le_minus2", "current_like_event"])),
          ("Surprise event (surprise ≤ −20% and z ≤ −2) against the current-like event", OVL),
          ("Surprise events: where the shortfall comes from (history class of the relationship at month m)",
           pct(SRC, ["share_of_shortfall"]) if len(SRC) else SRC))
ans("3-month surprise event (≤ −20%, z ≤ −2): share of customer-months / share also current-like / current-like also surprise",
    f"{CS.surprise_event.mean():.1%} / {CS.loc[CS.surprise_event, 'current_like'].mean():.0%} / "
    f"{CS.loc[CS.current_like, 'surprise_event'].mean():.0%}")

## §6 · Surprise for the whole population (for the contagion study)

The same expected / came / variance for every incoming relationship of every TM organisation, as of each month from
`SURPRISE_FROM`, written per stream to `SURPRISE_OUT` with the event month e = m + 2. The contagion notebook (§5h)
reads it, leaves out hub, PNC-entity, related and own-account payers like its other rules, adds up per customer and
month, and tests "surprise ≥ 20% of usual income with z ≤ −2" as an 11th definition. BG/BB is applied through a
lookup table of every (segment, x, t_x, n), and the amount model as a Spark expression, so nothing leaves Spark.

In [ ]:
CELL.start("§6 · Surprise for the whole population")
if CONFIG["BUILD_SURPRISE_ALL"]:
    with step("BG/BB lookup table"):
        LK = bgbb_lookup(LAST_MI)
        LKs = F.broadcast(spark.createDataFrame(LK[["seg", "x", "t_x", "n", "p1", "e3", "vc3"]]))
        _segs = [k for k in BG if k != "all"]
    with step("grid, expectations and 3-month actuals for every TM organisation"):
        _f0 = mi_of(CONFIG["SURPRISE_FROM"]) - BASE
        GA = build_grid(A, _f0).where(F.col("y_amt3").isNotNull())
        _sg = F.concat(F.col("side"), F.lit(" · "), _band("med3p"))
        GA = (GA.withColumn("seg", F.when(_sg.isin(_segs), _sg).otherwise(F.lit("all")))
              .join(LKs, ["seg", "x", "t_x", "n"], "left")
              .withColumn("amt_if", amt_spark(AM))
              .withColumn("exp3", F.coalesce(F.col("e3"), F.lit(0.0)) * F.col("amt_if"))
              .withColumn("var3", F.col("amt_if") * F.col("amt_if")
                          * (F.coalesce(F.col("vc3"), F.lit(0.0)) + F.lit(float(np.exp(AM["s2"]) - 1)) * F.coalesce(F.col("e3"), F.lit(0.0))))
              .withColumn("act3", F.col("y_amt") + F.coalesce(F.col("y_amt2"), F.lit(0.0)) + F.coalesce(F.col("y_amt3"), F.lit(0.0)))
              .withColumn("steady", ((F.col("act") == 1) & (F.col("n6") >= 4) & (F.col("n12") >= 8)).cast("int"))
              .select("cust", "side", "other", (F.col("mi") + 2).alias("mi_e"), "exp3", "act3", "var3", "p1", "steady",
                      (F.col("s6") / 6.0).alias("usual"))
              .withColumn("month", F.concat_ws("-", ((F.col("mi_e") + BASE - 1) / 12).cast("int").cast("string"),
                                               F.lpad((((F.col("mi_e") + BASE - 1) % 12) + 1).cast("string"), 2, "0")))
              .where((F.col("exp3") > 0) | (F.col("act3") > 0)))
        GA.write.mode("overwrite").partitionBy("month").parquet(CONFIG["SURPRISE_OUT"])
        _out = spark.read.parquet(CONFIG["SURPRISE_OUT"])
        SA = (_out.groupBy("month").agg(F.count(F.lit(1)).alias("streams"), F.countDistinct("cust").alias("customers"),
                                        F.sum("exp3").alias("expected"), F.sum("act3").alias("came"))
              .orderBy("month").toPandas())
        SA["came_vs_expected"] = SA.came / SA.expected
    CELL.show((f"Written to {CONFIG['SURPRISE_OUT']} (event month = as-of month + 2)", pct(SA, ["came_vs_expected"])))
    ans("Whole-population surprise table: months / came ÷ expected (median over months)",
        f"{SA.month.min()} .. {SA.month.max()} / {SA.came_vs_expected.median():.2f}")
else:
    CELL.show(("§6", pd.DataFrame([dict(status="skipped (BUILD_SURPRISE_ALL = False)")])))

## §7 · Answer sheet

In [ ]:
CELL.start("§7 · Answer sheet")
AN = pd.DataFrame(ANSWERS)
AN.to_csv(os.path.join(OUT_DIR, "esb_answers.csv"), index=False)
CELL.show(("Answers", AN))